# silver.players + player_game_roster population

Cleans and normalizes roster data from the bronze play-by-play payload, then loads two Silver tables:

- `dbw_hockey_lakehouse.silver.players`
- `dbw_hockey_lakehouse.silver.player_game_roster`

A small DQ split is included so obviously bad roster rows do not get merged into the main Silver tables.


In [ ]:

# --------------------------------------------------
# Silver roster / player load
#
# What this notebook does:
# - reads roster data from bronze.play_by_play_raw
# - keeps only games whose roster data has not been loaded yet
# - parses rosterSpots into one row per player per game
# - standardizes player fields for Silver
# - splits obvious bad rows out before merge
# - merges into silver.players and silver.player_game_roster
#
# Write pattern:
# - incremental
# - merge / upsert
#
# Notes:
# - this notebook is a little more practical than the Gold layer on purpose
# - roster/player logic is kept separate from the event notebook since the grain is different
# --------------------------------------------------

from pyspark.sql import functions as F

BRONZE_RAW = "dbw_hockey_lakehouse.bronze.play_by_play_raw"
SILVER_SCHEMA = "dbw_hockey_lakehouse.silver"
SILVER_PLAYERS = "dbw_hockey_lakehouse.silver.players"
SILVER_PLAYER_GAME_ROSTER = "dbw_hockey_lakehouse.silver.player_game_roster"
SILVER_ROSTER_BAD = "dbw_hockey_lakehouse.silver.roster_bad_records"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")


In [ ]:

# --------------------------------------------------
# Create target tables if needed
# --------------------------------------------------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_PLAYERS} (
  player_id BIGINT,
  first_name STRING,
  last_name STRING,
  position_code STRING,
  headshot_url STRING,
  last_seen_season BIGINT,
  updated_at TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_PLAYER_GAME_ROSTER} (
  season BIGINT,
  game_id BIGINT,
  game_date DATE,
  player_id BIGINT,
  team_id BIGINT,
  sweater_number INT,
  position_code STRING,
  headshot_url STRING,
  source_url STRING,
  ingested_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (season)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_ROSTER_BAD} (
  season BIGINT,
  game_id BIGINT,
  player_id BIGINT,
  team_id BIGINT,
  dq_errors ARRAY<STRING>,
  source_url STRING,
  ingested_at TIMESTAMP,
  updated_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (season)
""")


In [ ]:

# --------------------------------------------------
# Load bronze data and keep only games we have not processed yet
# --------------------------------------------------

bronze_df = spark.table(BRONZE_RAW)

# Roster incrementality should be based on the roster bridge, not silver.events.
# Different table, different grain.
processed_roster_games = (
    spark.table(SILVER_PLAYER_GAME_ROSTER)
    .select("game_id")
    .distinct()
)

bronze_incremental = bronze_df.join(processed_roster_games, on="game_id", how="left_anti")

print("bronze rows:", bronze_df.count())
print("incremental roster rows:", bronze_incremental.count())
display(bronze_incremental.select("game_id", "season", "source_url").limit(10))


In [ ]:

# --------------------------------------------------
# Quick source check
# --------------------------------------------------

# Basic sanity check so we know rosterSpots is actually present in the raw payload.
diag = (
    bronze_incremental
    .select(
        "game_id",
        F.get_json_object("raw_json", "$.rosterSpots").alias("roster_spots_json")
    )
    .withColumn("has_roster_spots", F.col("roster_spots_json").isNotNull())
)

display(diag.groupBy("has_roster_spots").count())


In [ ]:

# --------------------------------------------------
# Parse raw JSON
# --------------------------------------------------

schema_row = (
    bronze_incremental
    .select(F.schema_of_json(F.col("raw_json")).alias("schema"))
    .where(F.col("raw_json").isNotNull())
    .limit(1)
    .collect()
)

if len(schema_row) == 0:
    print("No rows available to infer schema for roster/players. Skipping this run.")
    parsed_df = None
else:
    schema_str = schema_row[0]["schema"]
    parsed_df = bronze_incremental.withColumn("pbp", F.from_json(F.col("raw_json"), schema_str))
    display(parsed_df.select("game_id", "season", "source_url").limit(10))


In [ ]:

# --------------------------------------------------
# Build roster_df
# --------------------------------------------------

if parsed_df is None:
    print("Skipping roster_df build (no parsed_df).")
else:
    # firstName / lastName can come through as either plain strings or structs in this feed.
    # Handle both here so the downstream Silver tables stay consistent.
    roster_schema = (
        "array<struct<"
        "teamId:bigint,"
        "playerId:bigint,"
        "sweaterNumber:int,"
        "positionCode:string,"
        "headshot:string,"
        "firstName:variant,"
        "lastName:variant"
        ">>"
    )

    first_name_default = F.get_json_object(F.to_json(F.col("rs.firstName")), "$.default")
    last_name_default = F.get_json_object(F.to_json(F.col("rs.lastName")), "$.default")

    # When the source sends a raw JSON string, cast(string) can leave quotes behind.
    # Small cleanup here.
    first_name_raw = F.regexp_replace(F.col("rs.firstName").cast("string"), '^"|"$', "")
    last_name_raw = F.regexp_replace(F.col("rs.lastName").cast("string"), '^"|"$', "")

    roster_df = (
        parsed_df
        .select(
            F.col("game_id").cast("long").alias("game_id"),
            F.col("season").cast("long").alias("season"),
            F.col("source_url"),
            F.col("ingested_at"),
            F.to_date(F.get_json_object(F.col("raw_json"), "$.gameDate")).alias("game_date"),
            F.from_json(F.get_json_object(F.col("raw_json"), "$.rosterSpots"), roster_schema).alias("roster_spots"),
        )
        .select(
            "season",
            "game_id",
            "game_date",
            "source_url",
            "ingested_at",
            F.explode_outer("roster_spots").alias("rs"),
        )
        .select(
            "season",
            "game_id",
            "game_date",
            "source_url",
            "ingested_at",
            F.col("rs.teamId").cast("long").alias("team_id"),
            F.col("rs.playerId").cast("long").alias("player_id"),
            F.col("rs.sweaterNumber").cast("int").alias("sweater_number"),
            F.col("rs.positionCode").cast("string").alias("position_code"),
            F.coalesce(first_name_default, first_name_raw).alias("first_name"),
            F.coalesce(last_name_default, last_name_raw).alias("last_name"),
            F.col("rs.headshot").cast("string").alias("headshot_url"),
        )
    )

    display(roster_df.limit(20))


In [ ]:

# --------------------------------------------------
# Data quality split
# --------------------------------------------------

if parsed_df is None:
    print("Skipping DQ split (no parsed_df).")
else:
    dq_roster = (
        roster_df
        .withColumn(
            "dq_errors",
            F.array_remove(
                F.array(
                    F.when(F.col("game_id").isNull(), F.lit("missing_game_id")),
                    F.when(F.col("season").isNull(), F.lit("missing_season")),
                    F.when(F.col("team_id").isNull(), F.lit("missing_team_id")),
                    F.when(F.col("player_id").isNull(), F.lit("missing_player_id")),
                    F.when(F.col("position_code").isNull(), F.lit("missing_position_code")),
                    F.when(
                        F.col("first_name").isNull() | (F.length(F.col("first_name")) == 0),
                        F.lit("missing_first_name"),
                    ),
                    F.when(
                        F.col("last_name").isNull() | (F.length(F.col("last_name")) == 0),
                        F.lit("missing_last_name"),
                    ),
                ),
                F.lit(None),
            ),
        )
        .withColumn("dq_errors", F.coalesce(F.col("dq_errors"), F.array().cast("array<string>")))
    )

    # Keep the bad rows around instead of silently dropping them.
    # Helps catch bad upstream data without blocking the whole load.
    bad_roster = dq_roster.filter(F.size("dq_errors") > 0)
    good_roster = dq_roster.filter(F.size("dq_errors") == 0)

    bad_roster_stage = (
        bad_roster
        .select("season", "game_id", "player_id", "team_id", "dq_errors", "source_url", "ingested_at")
        .dropDuplicates(["game_id", "player_id"])
    )

    bad_roster_stage.createOrReplaceTempView("staging_roster_bad_records")

    print("good_roster rows:", good_roster.count())
    print("bad_roster rows:", bad_roster.count())
    display(bad_roster.limit(20))


In [ ]:

# --------------------------------------------------
# Build staging tables
# --------------------------------------------------

if parsed_df is None:
    print("Skipping staging build (no parsed_df).")
else:
    # player_game_roster stays at one row per player per game.
    # Drop duplicate game_id + player_id pairs so merge keys stay clean.
    roster_stage = (
        good_roster
        .select(
            F.col("season").cast("long").alias("season"),
            F.col("game_id").cast("long").alias("game_id"),
            F.col("game_date").cast("date").alias("game_date"),
            F.col("player_id").cast("long").alias("player_id"),
            F.col("team_id").cast("long").alias("team_id"),
            F.col("sweater_number").cast("int").alias("sweater_number"),
            F.col("position_code").cast("string").alias("position_code"),
            F.col("headshot_url").cast("string").alias("headshot_url"),
            F.col("source_url").cast("string").alias("source_url"),
            F.col("ingested_at").cast("timestamp").alias("ingested_at"),
        )
        .dropna(subset=["game_id", "player_id"])
        .dropDuplicates(["game_id", "player_id"])
        .withColumn("updated_at", F.current_timestamp())
    )

    # players is a lighter dimension keyed by player_id.
    # Keep the latest basic descriptive attributes and a last-seen season.
    players_stage = (
        good_roster
        .select(
            F.col("player_id").cast("long").alias("player_id"),
            F.col("first_name").cast("string").alias("first_name"),
            F.col("last_name").cast("string").alias("last_name"),
            F.col("position_code").cast("string").alias("position_code"),
            F.col("headshot_url").cast("string").alias("headshot_url"),
            F.col("season").cast("long").alias("last_seen_season"),
            F.col("ingested_at").cast("timestamp").alias("updated_at"),
        )
        .dropna(subset=["player_id"])
        .dropDuplicates(["player_id"])
    )

    roster_stage.createOrReplaceTempView("staging_player_game_roster")
    players_stage.createOrReplaceTempView("staging_players")

    print("staging_player_game_roster rows:", roster_stage.count())
    print("staging_players rows:", players_stage.count())


In [ ]:

# --------------------------------------------------
# Merge bad roster rows
# --------------------------------------------------

if parsed_df is None:
    print("Skipping bad roster merge (no parsed_df).")
else:
    spark.sql(f"""
    MERGE INTO {SILVER_ROSTER_BAD} t
    USING staging_roster_bad_records s
    ON t.game_id = s.game_id AND t.player_id = s.player_id
    WHEN MATCHED THEN UPDATE SET
      t.season = s.season,
      t.team_id = s.team_id,
      t.dq_errors = s.dq_errors,
      t.source_url = s.source_url,
      t.ingested_at = s.ingested_at,
      t.updated_at = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
      season, game_id, player_id, team_id, dq_errors, source_url, ingested_at, updated_at
    ) VALUES (
      s.season, s.game_id, s.player_id, s.team_id, s.dq_errors, s.source_url, s.ingested_at, current_timestamp()
    )
    """)


In [ ]:

# --------------------------------------------------
# Merge players dimension
# --------------------------------------------------

if parsed_df is None:
    print("Skipping players merge (no parsed_df).")
else:
    spark.sql(f"""
    MERGE INTO {SILVER_PLAYERS} t
    USING staging_players s
    ON t.player_id = s.player_id
    WHEN MATCHED THEN UPDATE SET
      t.first_name = COALESCE(s.first_name, t.first_name),
      t.last_name = COALESCE(s.last_name, t.last_name),
      t.position_code = COALESCE(s.position_code, t.position_code),
      t.headshot_url = COALESCE(s.headshot_url, t.headshot_url),
      t.last_seen_season = GREATEST(t.last_seen_season, s.last_seen_season),
      t.updated_at = GREATEST(t.updated_at, s.updated_at)
    WHEN NOT MATCHED THEN INSERT (
      player_id, first_name, last_name, position_code, headshot_url, last_seen_season, updated_at
    ) VALUES (
      s.player_id, s.first_name, s.last_name, s.position_code, s.headshot_url, s.last_seen_season, s.updated_at
    )
    """)

    display(spark.table(SILVER_PLAYERS).orderBy(F.col("player_id").desc()).limit(20))


In [ ]:

# --------------------------------------------------
# Merge player_game_roster bridge
# --------------------------------------------------

if parsed_df is None:
    print("Skipping roster merge (no parsed_df).")
else:
    spark.sql(f"""
    MERGE INTO {SILVER_PLAYER_GAME_ROSTER} t
    USING staging_player_game_roster s
    ON t.game_id = s.game_id AND t.player_id = s.player_id
    WHEN MATCHED THEN UPDATE SET
      t.season = s.season,
      t.game_date = COALESCE(s.game_date, t.game_date),
      t.team_id = COALESCE(s.team_id, t.team_id),
      t.sweater_number = COALESCE(s.sweater_number, t.sweater_number),
      t.position_code = COALESCE(s.position_code, t.position_code),
      t.headshot_url = COALESCE(s.headshot_url, t.headshot_url),
      t.source_url = COALESCE(s.source_url, t.source_url),
      t.ingested_at = COALESCE(s.ingested_at, t.ingested_at),
      t.updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT (
      season, game_id, game_date, player_id, team_id,
      sweater_number, position_code, headshot_url, source_url, ingested_at, updated_at
    ) VALUES (
      s.season, s.game_id, s.game_date, s.player_id, s.team_id,
      s.sweater_number, s.position_code, s.headshot_url, s.source_url, s.ingested_at, s.updated_at
    )
    """)

    display(spark.table(SILVER_PLAYER_GAME_ROSTER).orderBy(F.col("game_id").desc()).limit(20))


In [ ]:

# --------------------------------------------------
# Validation queries
# --------------------------------------------------

# Trade-proof check:
# if a player appears for more than one team in a season, this bridge should still
# hold the game-level team assignment correctly.
spark.sql(f"""
SELECT
  season,
  player_id,
  team_id,
  COUNT(DISTINCT game_id) AS games_played
FROM {SILVER_PLAYER_GAME_ROSTER}
GROUP BY season, player_id, team_id
ORDER BY games_played DESC
LIMIT 50
""").show(truncate=False)

display(spark.sql(f"SELECT COUNT(*) AS players FROM {SILVER_PLAYERS}"))
display(spark.sql(f"SELECT COUNT(*) AS roster_rows FROM {SILVER_PLAYER_GAME_ROSTER}"))

display(spark.sql(f"""
SELECT AVG(players_in_game) AS avg_players_per_game
FROM (
  SELECT game_id, COUNT(*) AS players_in_game
  FROM {SILVER_PLAYER_GAME_ROSTER}
  GROUP BY game_id
)
"""))
